# NLP Week 2 Assignment: Word Segmentation, POS Tagging & Sequence Labeling

**ชื่อ-นามสกุล:** ภูริพัฒน์ เหมกุล  
**รหัสนักศึกษา:** 6752300313  

This notebook contains all **DIY exercises** from Week 2, along with the necessary prerequisite functions and data structures.


## 0. Prerequisites & Helper Functions
Run this cell first to initialize the required dictionaries, models, and utility functions.

In [1]:
import math

# ---- Prerequisite 1: Thai Dictionary & MaxMatch Algorithms ----
THAI_DICT = {
    "ตา",        # eye
    "ตาก",       # to air out / dry in the sun
    "กลม",       # round
    "ลม",        # wind
    "นอน",       # to lie down / sleep
    "หลับ",      # asleep
    "นอนหลับ",   # to sleep (compound)
    "อากาศ",     # weather / air
    "นั่ง",       # to sit
    "ริม",        # edge / bank
    "ทะเล",      # sea
}

def forward_max_match(text, dictionary, max_word_len=8, verbose=False):
    """Forward Maximum Matching (FMM): greedy longest match, left to right."""
    tokens, steps = [], []
    i = 0
    while i < len(text):
        matched = None
        for L in range(min(max_word_len, len(text) - i), 0, -1):
            candidate = text[i:i + L]
            if candidate in dictionary:
                matched = candidate
                steps.append((i, candidate, True))
                break
            steps.append((i, candidate, False))
        if matched is None:
            matched = text[i]
            steps.append((i, matched, None))
        tokens.append(matched)
        i += len(matched)

    if verbose:
        print(f"FMM on {text!r}")
        for pos, cand, hit in steps:
            mark = "MATCH -> emit" if hit else ("no match -> emit single char" if hit is None else "not in dict")
            print(f"   i={pos:>2}  try {cand!r:<12} (len {len(cand)})  {mark}")
        print(f"   result: {'|'.join(tokens)}\n")
    return tokens

def backward_max_match(text, dictionary, max_word_len=8, verbose=False):
    """Backward Maximum Matching (BMM): greedy longest match, right to left."""
    tokens, steps = [], []
    j = len(text)
    while j > 0:
        matched = None
        for L in range(min(max_word_len, j), 0, -1):
            candidate = text[j - L:j]
            if candidate in dictionary:
                matched = candidate
                steps.append((j, candidate, True))
                break
            steps.append((j, candidate, False))
        if matched is None:
            matched = text[j - 1]
            steps.append((j, matched, None))
        tokens.insert(0, matched)
        j -= len(matched)

    if verbose:
        print(f"BMM on {text!r}")
        for pos, cand, hit in steps:
            mark = "MATCH -> emit" if hit else ("no match -> emit single char" if hit is None else "not in dict")
            print(f"   j={pos:>2}  try {cand!r:<12} (len {len(cand)})  {mark}")
        print(f"   result: {'|'.join(tokens)}\n")
    return tokens

# ---- Prerequisite 2: Unigram Language Model ----
N_TOKENS = 1000000
UNIGRAM_COUNTS = {
    "ตา":     4200,   # eye        - common
    "กลม":    1800,   # round      - common
    "ตาก":     320,   # to air out - rarer
    "ลม":     3600,   # wind       - common
    "นั่ง":    2000,   # to sit
    "นอน":    5100,
    "หลับ":   1400,
    "นอนหลับ": 900,
    "อากาศ":  2900,
}

def unigram_prob(word):
    return UNIGRAM_COUNTS.get(word, 1) / N_TOKENS

def score_segmentation(tokens):
    """Return (probability, log10 probability) of a segmentation under the unigram LM."""
    p = 1.0
    for w in tokens:
        p *= unigram_prob(w)
    return p, math.log10(p)

# ---- Prerequisite 3: HMM Transition and Emission Matrices ----
TAGS = ["NNP", "MD", "VB", "JJ", "NN", "RB", "DT"]

A = {
    "<s>": {"NNP": 0.2767, "MD": 0.0006, "VB": 0.0031, "JJ": 0.0453, "NN": 0.0449, "RB": 0.0510, "DT": 0.2026},
    "NNP": {"NNP": 0.3777, "MD": 0.0110, "VB": 0.0009, "JJ": 0.0084, "NN": 0.0584, "RB": 0.0090, "DT": 0.0025},
    "MD":  {"NNP": 0.0008, "MD": 0.0002, "VB": 0.7968, "JJ": 0.0005, "NN": 0.0008, "RB": 0.1698, "DT": 0.0041},
    "VB":  {"NNP": 0.0322, "MD": 0.0005, "VB": 0.0050, "JJ": 0.0837, "NN": 0.0615, "RB": 0.0514, "DT": 0.2231},
    "JJ":  {"NNP": 0.0366, "MD": 0.0004, "VB": 0.0001, "JJ": 0.0733, "NN": 0.4509, "RB": 0.0036, "DT": 0.0036},
    "NN":  {"NNP": 0.0096, "MD": 0.0176, "VB": 0.0014, "JJ": 0.0086, "NN": 0.1216, "RB": 0.0177, "DT": 0.0068},
    "RB":  {"NNP": 0.0068, "MD": 0.0102, "VB": 0.1011, "JJ": 0.1012, "NN": 0.0120, "RB": 0.0728, "DT": 0.0479},
    "DT":  {"NNP": 0.1147, "MD": 0.0021, "VB": 0.0002, "JJ": 0.2157, "NN": 0.4744, "RB": 0.0102, "DT": 0.0017},
}

B = {
    "NNP": {"Janet": 0.000032, "will": 0.0,      "back": 0.0,      "the": 0.000048, "bill": 0.0},
    "MD":  {"Janet": 0.0,      "will": 0.308431, "back": 0.0,      "the": 0.0,      "bill": 0.0},
    "VB":  {"Janet": 0.0,      "will": 0.000028, "back": 0.000672, "the": 0.0,      "bill": 0.000028},
    "JJ":  {"Janet": 0.0,      "will": 0.0,      "back": 0.000340, "the": 0.0,      "bill": 0.0},
    "NN":  {"Janet": 0.0,      "will": 0.000200, "back": 0.000223, "the": 0.0,      "bill": 0.002337},
    "RB":  {"Janet": 0.0,      "will": 0.0,      "back": 0.010446, "the": 0.0,      "bill": 0.0},
    "DT":  {"Janet": 0.0,      "will": 0.0,      "back": 0.0,      "the": 0.506099, "bill": 0.0},
}

# ---- Prerequisite 4: BIO and Sequence Labeling Utilities ----
def spans_to_bio(tokens, spans):
    """spans: list of (start, end_inclusive, type) -> BIO tag list."""
    tags = ["O"] * len(tokens)
    for start, end, etype in spans:
        tags[start] = f"B-{etype}"
        for i in range(start + 1, end + 1):
            tags[i] = f"I-{etype}"
    return tags

def bio_to_spans(tags):
    """BIO tag list -> list of (start, end_inclusive, type)."""
    spans, start, etype = [], None, None
    for i, tag in enumerate(tags):
        if tag.startswith("B-"):
            if start is not None:
                spans.append((start, i - 1, etype))
            start, etype = i, tag[2:]
        elif tag.startswith("I-") and start is not None and tag[2:] == etype:
            continue
        else:
            if start is not None:
                spans.append((start, i - 1, etype))
            start, etype = None, None
    if start is not None:
        spans.append((start, len(tags) - 1, etype))
    return spans

def bio_to_io(tags):
    return [t.replace("B-", "I-") for t in tags]

def bio_to_bioes(tags):
    """B-X I-X I-X -> B-X I-X E-X ;  a lone B-X -> S-X"""
    spans = bio_to_spans(tags)
    out = ["O"] * len(tags)
    for start, end, etype in spans:
        if start == end:
            out[start] = f"S-{etype}"
        else:
            out[start] = f"B-{etype}"
            for i in range(start + 1, end):
                out[i] = f"I-{etype}"
            out[end] = f"E-{etype}"
    return out

print("Setup completed: All prerequisite functions and variables loaded successfully.")


Setup completed: All prerequisite functions and variables loaded successfully.


---
## Part 1: Word Segmentation


---
### 🔧 DIY 1 — Trace Maximum Matching yourself

The string **อากาศตากลม** ("the weather/air, airing in the wind") extends our ambiguous case with a preceding word.

1. Predict on paper what FMM and BMM will output, using `THAI_DICT`.
2. Then complete the code below to check your prediction.
3. Answer in the variables provided: does the boundary between ตาก/ตา land in the same place for both?

In [2]:
# === DIY 1 ===
text = "อากาศตากลม"

# TODO 1: run forward_max_match on 'text' with THAI_DICT and verbose=True
fmm_out = forward_max_match(text, THAI_DICT, verbose=True)

# TODO 2: run backward_max_match on 'text' with THAI_DICT and verbose=True
bmm_out = backward_max_match(text, THAI_DICT, verbose=True)

# TODO 3: set this to True if the two algorithms produced DIFFERENT segmentations
they_disagree = (fmm_out != bmm_out)

print("FMM:", fmm_out)
print("BMM:", bmm_out)
print("Disagree?", they_disagree)


FMM on 'อากาศตากลม'
   i= 0  try 'อากาศตาก'   (len 8)  not in dict
   i= 0  try 'อากาศตา'    (len 7)  not in dict
   i= 0  try 'อากาศต'     (len 6)  not in dict
   i= 0  try 'อากาศ'      (len 5)  MATCH -> emit
   i= 5  try 'ตากลม'      (len 5)  not in dict
   i= 5  try 'ตากล'       (len 4)  not in dict
   i= 5  try 'ตาก'        (len 3)  MATCH -> emit
   i= 8  try 'ลม'         (len 2)  MATCH -> emit
   result: อากาศ|ตาก|ลม

BMM on 'อากาศตากลม'
   j=10  try 'กาศตากลม'   (len 8)  not in dict
   j=10  try 'าศตากลม'    (len 7)  not in dict
   j=10  try 'ศตากลม'     (len 6)  not in dict
   j=10  try 'ตากลม'      (len 5)  not in dict
   j=10  try 'ากลม'       (len 4)  not in dict
   j=10  try 'กลม'        (len 3)  MATCH -> emit
   j= 7  try 'อากาศตา'    (len 7)  not in dict
   j= 7  try 'ากาศตา'     (len 6)  not in dict
   j= 7  try 'กาศตา'      (len 5)  not in dict
   j= 7  try 'าศตา'       (len 4)  not in dict
   j= 7  try 'ศตา'        (len 3)  not in dict
   j= 7  try 'ตา'         (len 2) 

---
### 🔧 DIY 2 — Resolve an ambiguity with probabilities

The string **ลมกลม** could be segmented `ลม|กลม` ("round wind"). Suppose you add a new word to the corpus:

| word | count |
|---|---|
| ลมกลม (a hypothetical single word) | 50 |

Compute, using `unigram_prob`, whether the unigram model prefers the **one-word** analysis `["ลมกลม"]` or the **two-word** analysis `["ลม", "กลม"]`, and by what factor. Think first: a single rare word vs. two common words — which wins, and why?

In [3]:
# === DIY 2 ===
UNIGRAM_COUNTS["ลมกลม"] = 50      # register the hypothetical compound

one_word = ["ลมกลม"]
two_words = ["ลม", "กลม"]

# TODO 1: score both analyses with score_segmentation()
p_one, log_one = score_segmentation(one_word)
p_two, log_two = score_segmentation(two_words)

# TODO 2: which analysis wins, and by what factor?
winner = one_word if p_one > p_two else two_words
factor = max(p_one, p_two) / min(p_one, p_two)

print(f"one word : {p_one}")
print(f"two words: {p_two}")
print(f"winner   : {winner}  by {factor}x")


one word : 5e-05
two words: 6.48e-06
winner   : ['ลมกลม']  by 7.71604938271605x


---
## Part 2: POS Tagging & The Viterbi Algorithm


---
### 🔧 DIY 3 — Compute a Viterbi cell by hand

Compute $v_2(\text{NN})$ — the best path probability for tagging *will* as a common noun — **on paper first**, then verify.

You need: $v_1(\text{NNP}) = 8.8544 \times 10^{-6}$, $a_{\text{NNP},\text{NN}}$ from Figure 18.12, and $b_{\text{NN}}(\text{will})$ from Figure 18.13. (Every other predecessor is 0, so the max has exactly one non-zero candidate.)

Then answer: how many times larger is $v_2(\text{MD})$ than $v_2(\text{NN})$, and what does that ratio *mean* linguistically?

In [4]:
# === DIY 3 ===
v1_nnp = A["<s>"]["NNP"] * B["NNP"]["Janet"]     # given: 8.8544e-6

# TODO 1: look up the two probabilities you need
a_nnp_nn = A["NNP"]["NN"]     # <-- P(NN | NNP) from Figure 18.12
b_nn_will = B["NN"]["will"]     # <-- P(will | NN) from Figure 18.13

# TODO 2: compute v2(NN)
v2_nn = v1_nnp * a_nnp_nn * b_nn_will

# TODO 3: how many times larger is v2(MD) than v2(NN)?
v2_md = v1_nnp * A["NNP"]["MD"] * B["MD"]["will"]
ratio = v2_md / v2_nn if v2_nn != 0 else float('inf')

print(f"v2(NN) = {v2_nn}")
print(f"v2(MD) = {v2_md:.6e}")
print(f"MD is {ratio:.4f} times more likely than NN")


v2(NN) = 1.03419392e-10
v2(MD) = 3.004069e-08
MD is 290.4744 times more likely than NN


---
## Part 3: Named Entity Recognition & Evaluation


---
### 🔧 DIY 4 — BIO-tag a Thai sentence

Take the sentence **"บริษัทปูนซิเมนต์ไทยตั้งอยู่ที่กรุงเทพมหานคร"** ("The Siam Cement Company is located in Bangkok"), pre-segmented for you below.

Gold entities: **ปูนซิเมนต์ไทย** (tokens 1–2, `ORG`) and **กรุงเทพมหานคร** (token 5, `LOC`).

1. Write the BIO tags.
2. Convert them to BIOES.
3. Verify with `bio_to_spans` that you can recover the entities.

In [5]:
# === DIY 4 ===
th_tokens = ["บริษัท", "ปูนซิเมนต์", "ไทย", "ตั้งอยู่", "ที่", "กรุงเทพมหานคร"]
#   index:       0          1          2       3        4          5
indexed_tokens = list(enumerate(th_tokens))
# gold entities: (1,2,'ORG')  and  (5,5,'LOC')
gold_spans = [(1, 2, "ORG"), (5, 5, "LOC")]

# TODO 1: write the BIO tags by hand (6 strings)
th_bio = ["O", "B-ORG", "I-ORG", "O", "O", "B-LOC"]

# TODO 2: convert to BIOES using bio_to_bioes()
th_bioes = bio_to_bioes(th_bio)

# TODO 3: recover the spans and check they match the gold entities
recovered = bio_to_spans(th_bio)

print("BIO  :", th_bio)
print("BIOES:", th_bioes)
print("spans:", recovered)


BIO  : ['O', 'B-ORG', 'I-ORG', 'O', 'O', 'B-LOC']
BIOES: ['O', 'B-ORG', 'E-ORG', 'O', 'O', 'S-LOC']
spans: [(1, 2, 'ORG'), (5, 5, 'LOC')]


---
### 🔧 DIY 5 — Compute Precision, Recall, and F1

A system is run on a small test set. Here are the gold and predicted entity sets:

| | Entities |
|---|---|
| **Gold** | *สมชาย*/PER, *เชียงใหม่*/LOC, *ปตท.*/ORG, *กรุงเทพ*/LOC, *2026*/DATE |
| **Predicted** | *สมชาย*/PER, *เชียง*/LOC, *ปตท.*/ORG, *กรุงเทพ*/ORG |

Compute TP, FP, FN, then Precision, Recall, and F1 — **by hand first**, then check with code. Also: is precision or recall the bigger problem for this system, and what does that suggest you fix?

In [6]:
# === DIY 5 ===
gold_set = {("สมชาย", "PER"), ("เชียงใหม่", "LOC"), ("ปตท.", "ORG"),
            ("กรุงเทพ", "LOC"), ("2026", "DATE")}
pred_set = {("สมชาย", "PER"), ("เชียง", "LOC"), ("ปตท.", "ORG"),
            ("กรุงเทพ", "ORG")}

tp_set = gold_set & pred_set
fp_set = pred_set - gold_set
fn_set = gold_set - pred_set

# TODO 1: compute the three counts
tp = len(tp_set)
fp = len(fp_set)
fn = len(fn_set)

# TODO 2: compute the three metrics
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print(f"TP={tp} FP={fp} FN={fn}")
print(f"P={precision:.4f} R={recall:.4f} F1={f1:.4f}")


TP=2 FP=2 FN=3
P=0.5000 R=0.4000 F1=0.4444
